# 加法进位实验


<img src="https://github.com/JerrikEph/jerrikeph.github.io/raw/master/Learn2Carry.png" width=650>

In [ ]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
# from tensorflow.keras import layers
# from tensorflow.keras import layers, optimizers, datasets
# 注意：当前环境是Python3.11.15+TF2 ，上面这句会报错，统一改用 `keras.xxx`
layers = keras.layers
optimizers = keras.optimizers
datasets = keras.datasets
import os,sys


## 数据生成
我们随机在 `start->end`之间采样除整数对`(num1, num2)`，计算结果`num1+num2`作为监督信号。

* 首先将数字转换成数字位列表 `convertNum2Digits`
* 将数字位列表反向
* 将数字位列表填充到同样的长度 `pad2len`


In [31]:
def gen_data_batch(batch_size, start, end):
    '''在(start, end)区间采样生成一个batch的整型的数据
    Args :
        batch_size: batch_size
        start: 开始数值
        end: 结束数值
    '''
    numbers_1 = np.random.randint(start, end, batch_size)
    numbers_2 = np.random.randint(start, end, batch_size)
    results = numbers_1 + numbers_2
    return numbers_1, numbers_2, results

def convertNum2Digits(Num):
    '''将一个整数转换成一个数字位的列表,例如 133412 ==> [1, 3, 3, 4, 1, 2]
    '''
    strNum = str(Num)
    chNums = list(strNum)
    digitNums = [int(o) for o in strNum]
    return digitNums

def convertDigits2Num(Digits):
    '''将数字位列表反向， 例如 [1, 3, 3, 4, 1, 2] ==> [2, 1, 4, 3, 3, 1]
    '''
    digitStrs = [str(o) for o in Digits]
    numStr = ''.join(digitStrs)
    Num = int(numStr)
    return Num

def pad2len(lst, length, pad=0):
    '''将一个列表用`pad`填充到`length`的长度 例如 pad2len([1, 3, 2, 3], 6, pad=0) ==> [1, 3, 2, 3, 0, 0]
    '''
    lst+=[pad]*(length - len(lst))
    return lst

def results_converter(res_lst):
    '''将预测好的数字位列表批量转换成为原始整数
    Args:
        res_lst: shape(b_sz, len(digits))
    '''
    res = [reversed(digits) for digits in res_lst]
    return [convertDigits2Num(digits) for digits in res]

def prepare_batch(Nums1, Nums2, results, maxlen):
    '''准备一个batch的数据，将数值转换成反转的数位列表并且填充到固定长度
    Args:
        Nums1: shape(batch_size,)
        Nums2: shape(batch_size,)
        results: shape(batch_size,)
        maxlen:  type(int)
    Returns:
        Nums1: shape(batch_size, maxlen)
        Nums2: shape(batch_size, maxlen)
        results: shape(batch_size, maxlen)
    '''
    Nums1 = [convertNum2Digits(o) for o in Nums1]
    Nums2 = [convertNum2Digits(o) for o in Nums2]
    results = [convertNum2Digits(o) for o in results]
    
    Nums1 = [list(reversed(o)) for o in Nums1]
    Nums2 = [list(reversed(o)) for o in Nums2]
    results = [list(reversed(o)) for o in results]
    
    Nums1 = [pad2len(o, maxlen) for o in Nums1]
    Nums2 = [pad2len(o, maxlen) for o in Nums2]
    results = [pad2len(o, maxlen) for o in results]
    
    return Nums1, Nums2, results

# 建模过程， 按照图示完成建模

In [33]:
class myRNNModel(keras.Model):
    def __init__(self):
        super(myRNNModel, self).__init__()
        # Keras 3 中 Embedding 不再使用 batch_input_shape 参数
        self.embed_layer = keras.layers.Embedding(10, 32)
        
        self.rnncell = keras.layers.SimpleRNNCell(64)
        self.rnn_layer = keras.layers.RNN(self.rnncell, return_sequences=True)
        self.dense = keras.layers.Dense(10)

    def call(self, num1, num2):
        '''
        此处完成上述图中模型
        '''
        # num1/num2: shape (batch, seq_len)
        emb1 = self.embed_layer(num1)               # 32
        emb2 = self.embed_layer(num2)               # 32
        # 按最后一维拼接
        inp_emb = tf.concat([emb1, emb2], axis=-1) # 64
        # RNN按位处理，学习进位依赖
        rnn_out = self.rnn_layer(inp_emb)           # 64
        # 每一位输出 0~9 的分类
        logits = self.dense(rnn_out)                # 10
        return logits

In [35]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    return tf.reduce_mean(losses)

@tf.function
def train_one_step(model, optimizer, x, y, label):
    with tf.GradientTape() as tape:
        logits = model(x, y)
        loss = compute_loss(logits, label)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(steps, model, optimizer):
    loss = 0.0
    accuracy = 0.0
    for step in range(steps):
        datas = gen_data_batch(batch_size=200, start=0, end=555555555)
        Nums1, Nums2, results = prepare_batch(*datas, maxlen=11)
        loss = train_one_step(model, optimizer, tf.constant(Nums1, dtype=tf.int32), 
                              tf.constant(Nums2, dtype=tf.int32),
                              tf.constant(results, dtype=tf.int32))
        if step%50 == 0:
            print('step', step, ': loss', loss.numpy())

    return loss

def evaluate(model):
    datas = gen_data_batch(batch_size=2000, start=555555555, end=999999999)
    Nums1, Nums2, results = prepare_batch(*datas, maxlen=11)
    logits = model(tf.constant(Nums1, dtype=tf.int32), tf.constant(Nums2, dtype=tf.int32))
    logits = logits.numpy()
    pred = np.argmax(logits, axis=-1)
    res = results_converter(pred)
    for o in list(zip(datas[2], res))[:20]:
        print(o[0], o[1], o[0]==o[1])

    print('accuracy is: %g' % np.mean([o[0]==o[1] for o in zip(datas[2], res)]))


In [36]:
optimizer = optimizers.Adam(0.001)
model = myRNNModel()

In [37]:
train(3000, model, optimizer)
evaluate(model)

step 0 : loss 2.3099546
step 50 : loss 1.9136785
step 100 : loss 1.899138
step 150 : loss 1.8917669
step 200 : loss 1.8834316
step 250 : loss 1.8867793
step 300 : loss 1.8850994
step 350 : loss 1.8757242
step 400 : loss 1.8731135
step 450 : loss 1.863681
step 500 : loss 1.8669941
step 550 : loss 1.8678977
step 600 : loss 1.8678957
step 650 : loss 1.8568757
step 700 : loss 1.8467531
step 750 : loss 1.8059906
step 800 : loss 1.7433175
step 850 : loss 1.619933
step 900 : loss 1.416036
step 950 : loss 1.1585232
step 1000 : loss 0.95993966
step 1050 : loss 0.8065475
step 1100 : loss 0.68095046
step 1150 : loss 0.5723823
step 1200 : loss 0.48961854
step 1250 : loss 0.4083836
step 1300 : loss 0.34743363
step 1350 : loss 0.28064325
step 1400 : loss 0.23247387
step 1450 : loss 0.19677481
step 1500 : loss 0.16492957
step 1550 : loss 0.13686398
step 1600 : loss 0.110900946
step 1650 : loss 0.09475236
step 1700 : loss 0.078029975
step 1750 : loss 0.06816276
step 1800 : loss 0.05854004
step 1850 : 